# Grouping Comparison Experiment

Compares five alert-grouping methods (fixed window, time-delta, CSCAS-style,
AlertBERT, DeepCASE) on the held-out test scenarios (`fox`,
`russellmitchell`): alert volume reduction, group size distribution, group
purity, and (DeepCASE only) the fraction of alerts rejected outright rather
than grouped (`ungroupable_fraction`).

Each method's sweep now runs as its own standalone script
(`src/thesis/baselines/grouping/*.py`) rather than inline in this notebook
-- this notebook only loads their `results/*.json` (+ `*_sizes.npz`)
artifacts and plots them. See `run_overnight.sh` in that directory to run
every script overnight; the `GROUPING_DEVICE` env var selects `mps`/`cuda`/
`cpu` (default: auto-detect) for the two torch-based methods.

**Run the scripts first** to generate the artifacts this notebook reads:
```
cd src/thesis/baselines/grouping
python fixed_window.py
python time_delta.py
python cscas_grouping.py
python cscas_grouping_sensitivity.py
python deepcase_sweep.py
```
`alertbert_sweep.py` needs the separate `thesis-alertbert` conda env
(`alertbert.models` hard-imports `graph_tool`, not part of this project's
plain `venv/`):
```
conda activate thesis-alertbert
KMP_DUPLICATE_LIB_OK=TRUE python alertbert_sweep.py
```
This notebook itself only needs the plain venv -- it never imports
`thesis.grouping`/`torch`/`alertbert`/`deepcase`.


## 1. Imports

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thesis.baselines.grouping._results import (
    load_grouping_results,
    load_group_size_arrays,
)


## 2. Settings

Edit and re-run -- nothing past this cell needs to change.


In [ ]:
# The 5 methods that feed the shared cross-method comparison (each has a
# results/{name}.json + results/{name}_sizes.npz, written by the
# like-named script in src/thesis/baselines/grouping/).
GROUPING_RESULT_NAMES = ["fixed_window", "time_delta", "cscas_grouping", "alertbert", "deepcase"]

# CSCAS's authors' own validated production settings -- must match the
# primary config baked into cscas_grouping.py/cscas_grouping_sensitivity.py.
# Only used here for the sensitivity plot's reference lines.
CSCAS_SESSION_LENGTH_SECONDS = 300.0
CSCAS_SESSION_TIMEOUT_SECONDS = 60.0


## 3. Loading

Tolerant of missing artifacts (prints `[skip] ...` and continues) -- run
whichever scripts you have, this notebook plots whatever's available.


In [ ]:
results_dfs: dict[str, pd.DataFrame] = {}
size_arrays: dict[tuple, np.ndarray] = {}
for name in GROUPING_RESULT_NAMES:
    try:
        results_dfs[name] = load_grouping_results(name)
        size_arrays.update(load_group_size_arrays(name))
    except FileNotFoundError as e:
        print(f"[skip] {e}")

print(f"Loaded results for: {list(results_dfs.keys())}")

# Derived columns needed by the per-method diagnostic plots below -- moved
# here from what used to be inline sweep cells.
if "time_delta" in results_dfs:
    results_dfs["time_delta"]["delta"] = results_dfs["time_delta"]["param"].astype(float)
if "deepcase" in results_dfs:
    results_dfs["deepcase"]["context_length"] = (
        results_dfs["deepcase"]["param"].str.extract(r"L=(\d+)").astype(int)
    )
    results_dfs["deepcase"]["eps"] = (
        results_dfs["deepcase"]["param"].str.extract(r"eps=([\d.]+)").astype(float)
    )

all_results_df = (
    pd.concat(results_dfs.values(), ignore_index=True) if results_dfs else pd.DataFrame()
)

try:
    cscas_sensitivity_df = load_grouping_results("cscas_grouping_sensitivity")
except FileNotFoundError as e:
    print(f"[skip] {e}")
    cscas_sensitivity_df = pd.DataFrame()

try:
    manual_review_df = load_grouping_results("deepcase_manual_review")
except FileNotFoundError as e:
    print(f"[skip] {e}")
    manual_review_df = pd.DataFrame()


## 4. Fixed window

Linear sweep (this method's parameter is an absolute group size, not a
gap threshold, so no log-scale sweep here — see the time-delta section
for why that distinction matters).


In [ ]:
fixed_window_df = results_dfs.get("fixed_window")
if fixed_window_df is None:
    print("[skip] fixed_window not loaded -- run fixed_window.py first")
else:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    for method_name, group in fixed_window_df.groupby("method"):
        agg = group.groupby("param", as_index=False)["reduction"].mean()
        agg["param"] = agg["param"].astype(int)
        agg = agg.sort_values("param")
        ax[0].plot(agg["param"], agg["reduction"], marker="o", label=method_name)

    ax[0].set_xlabel("Window size (s)")
    ax[0].set_ylabel("Alert volume reduction")
    ax[0].set_title("Fixed window: reduction vs window size")
    ax[0].legend()

    for method_name, group in fixed_window_df.groupby("method"):
        agg = group.groupby("param", as_index=False)["group_size_mean"].mean()
        agg["param"] = agg["param"].astype(int)
        agg = agg.sort_values("param")
        ax[1].plot(agg["param"], agg["group_size_mean"], marker="o", label=method_name)

    ax[1].set_xlabel("Window size (s)")
    ax[1].set_ylabel("Mean group size")
    ax[1].set_title("Fixed window: mean group size vs window size")
    ax[1].legend()

    plt.tight_layout()
    plt.show()


## 5. Time-delta

Log-scale sweep matching the original method's own validated evaluation
protocol: `delta = a * 2**i` for `i = -7 ... 13`, `a in {1, 1.5}`. This
range spans sub-second to multi-hour gaps, which matters specifically
because AIT-ADS alerts have whole-second timestamp resolution for many
detectors — a coarser, linear sweep would miss the discontinuity around
delta = 1s entirely.


In [ ]:
time_delta_df = results_dfs.get("time_delta")
if time_delta_df is None:
    print("[skip] time_delta not loaded -- run time_delta.py first")
else:
    # Per-method diagnostic plot: log-x reduction curve, flags the delta=1s
    # discontinuity described above.
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    for method_name, group in time_delta_df.groupby("method"):
        agg = group.groupby("delta", as_index=False)["reduction"].mean().sort_values("delta")
        ax[0].plot(agg["delta"], agg["reduction"], marker=".", label=method_name)

    ax[0].set_xscale("log")
    ax[0].axvline(1.0, color="grey", linestyle="--", linewidth=1, label="delta = 1s")
    ax[0].set_xlabel("delta (s, log scale)")
    ax[0].set_ylabel("Alert volume reduction")
    ax[0].set_title("Time-delta: reduction vs delta")
    ax[0].legend()

    for method_name, group in time_delta_df.groupby("method"):
        agg = group.groupby("delta", as_index=False)["group_size_mean"].mean().sort_values("delta")
        ax[1].plot(agg["delta"], agg["group_size_mean"], marker=".", label=method_name)

    ax[1].set_xscale("log")
    ax[1].set_yscale("log")
    ax[1].axvline(1.0, color="grey", linestyle="--", linewidth=1)
    ax[1].set_xlabel("delta (s, log scale)")
    ax[1].set_ylabel("Mean group size (log scale)")
    ax[1].set_title("Time-delta: mean group size vs delta")
    ax[1].legend()

    plt.tight_layout()
    plt.show()


## 6. CSCAS grouping

Single fixed configuration — the authors' own validated production
settings (SessionLength=300s, SessionTimeout=60s), not swept.


In [ ]:
cscas_df = results_dfs.get("cscas_grouping")
if cscas_df is None:
    print("[skip] cscas_grouping not loaded -- run cscas_grouping.py first")
else:
    # Per-method diagnostic plot: reduction and purity per scenario (single config, so bars not curves)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    ax[0].bar(cscas_df["scenario"], cscas_df["reduction"])
    ax[0].set_ylabel("Alert volume reduction")
    ax[0].set_title("CSCAS grouping: reduction per scenario")

    purity_cols = ["pure_benign_frac", "pure_attack_frac", "mixed_frac"]
    bottom = np.zeros(len(cscas_df))
    for col in purity_cols:
        ax[1].bar(cscas_df["scenario"], cscas_df[col], bottom=bottom, label=col)
        bottom += cscas_df[col].values
    ax[1].set_ylabel("Fraction of groups")
    ax[1].set_title("CSCAS grouping: group purity per scenario")
    ax[1].legend()

    plt.tight_layout()
    plt.show()


### 6b. CSCAS sensitivity sweep (secondary)

The run above uses the authors' single validated production configuration
(`SessionLength=300s`, `SessionTimeout=60s`) as CSCAS's primary result --
that's the artifact that feeds the cross-method comparison in section 9.
Those values were tuned by the original authors for Suricata's alert
density specifically, and CSCAS-style grouping has since been generalized
here to all three IDS sources -- so it's a fair question whether 60s/300s
still makes sense once the input population has changed.

This secondary sweep (`cscas_grouping_sensitivity.py`, a separate script
writing a separate artifact) answers that as a **robustness check, not a
search for a better operating point**: it varies each parameter one at a
time rather than a full factorial -- `session_timeout ∈ {30, 60, 120}` with
`session_length` held at its primary value (300), and
`session_length ∈ {150, 300, 600}` with `session_timeout` held at its
primary value (60), tagged `varied`/`varied_value` so the two grids can be
filtered and plotted independently below. This artifact never enters
`all_results_df` -- if it did, a sensitivity setting could compete with the
primary config for CSCAS's "best-reduction setting" slot in section 9's
plots, which would misrepresent CSCAS's operating point.


In [ ]:
if cscas_sensitivity_df.empty:
    print("[skip] cscas_grouping_sensitivity not loaded -- run cscas_grouping_sensitivity.py first")
else:
    # Sensitivity diagnostic: reduction vs. each parameter, one-at-a-time,
    # with the authors' validated value marked -- a robustness check (does
    # the published config still make sense here), not a search for a
    # better operating point, so the plot deliberately doesn't highlight
    # any other point as "best."
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    timeout_df = cscas_sensitivity_df[cscas_sensitivity_df["varied"] == "session_timeout"]
    for scenario in sorted(cscas_sensitivity_df["scenario"].unique()):
        scen_df = timeout_df[timeout_df["scenario"] == scenario].sort_values("varied_value")
        axes[0].plot(scen_df["varied_value"], scen_df["reduction"], marker="o", label=scenario)
    axes[0].axvline(
        CSCAS_SESSION_TIMEOUT_SECONDS, color="red", linestyle="--",
        label=f"authors' validated value ({CSCAS_SESSION_TIMEOUT_SECONDS:g}s)",
    )
    axes[0].set_xlabel("session_timeout (s)")
    axes[0].set_ylabel("Alert volume reduction")
    axes[0].set_title(f"Reduction vs session_timeout\n(session_length fixed at {CSCAS_SESSION_LENGTH_SECONDS:g}s)")
    axes[0].legend()

    length_df = cscas_sensitivity_df[cscas_sensitivity_df["varied"] == "session_length"]
    for scenario in sorted(cscas_sensitivity_df["scenario"].unique()):
        scen_df = length_df[length_df["scenario"] == scenario].sort_values("varied_value")
        axes[1].plot(scen_df["varied_value"], scen_df["reduction"], marker="o", label=scenario)
    axes[1].axvline(
        CSCAS_SESSION_LENGTH_SECONDS, color="red", linestyle="--",
        label=f"authors' validated value ({CSCAS_SESSION_LENGTH_SECONDS:g}s)",
    )
    axes[1].set_xlabel("session_length (s)")
    axes[1].set_ylabel("Alert volume reduction")
    axes[1].set_title(f"Reduction vs session_length\n(session_timeout fixed at {CSCAS_SESSION_TIMEOUT_SECONDS:g}s)")
    axes[1].legend()

    fig.suptitle("CSCAS sensitivity sweep (secondary) -- robustness check, not a search for a better operating point")
    plt.tight_layout()
    plt.show()


## 7. AlertBERT

Delta and theta swept separately, matching the AlertBERT paper's own
asymmetric design -- see `alertbert_sweep.py`'s module docstring for the full
grid rationale, checkpoint compatibility, and GPU/conda-env requirements.
No diagnostic plot here (none existed before this refactor either) -- its
rows feed directly into section 9's cross-method comparison.


## 8. DeepCASE

Two-tier sweep (context length x DBSCAN eps) -- see `deepcase_sweep.py`'s module
docstring for the full sweep design, outlier convention, and runtime notes.


In [ ]:
deepcase_df = results_dfs.get("deepcase")
if deepcase_df is None:
    print("[skip] deepcase not loaded -- run deepcase_sweep.py first")
else:
    # Per-method diagnostic plot: reduction and mean group size vs eps, one
    # line per context length L.
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))

    for L, group in deepcase_df.groupby("context_length"):
        agg = group.groupby("eps", as_index=False)["reduction"].mean().sort_values("eps")
        ax[0].plot(agg["eps"], agg["reduction"], marker=".", label=f"L={L}")

    ax[0].set_xlabel("eps")
    ax[0].set_ylabel("Alert volume reduction")
    ax[0].set_title("DeepCASE: reduction vs eps")
    ax[0].legend()

    for L, group in deepcase_df.groupby("context_length"):
        agg = group.groupby("eps", as_index=False)["group_size_mean"].mean().sort_values("eps")
        ax[1].plot(agg["eps"], agg["group_size_mean"], marker=".", label=f"L={L}")

    ax[1].set_yscale("log")
    ax[1].set_xlabel("eps")
    ax[1].set_ylabel("Mean group size (log scale)")
    ax[1].set_title("DeepCASE: mean group size vs eps")
    ax[1].legend()

    plt.tight_layout()
    plt.show()


### 8b. DeepCASE manual-review queue

Each outlier alert's `reason` (`"deepcase_low_confidence"` vs
`"deepcase_dbscan_noise"`) feeds `build_manual_review_records` (inside
`deepcase_sweep.py`), which turns the rejected alerts at each (scenario,
context_length)'s default eps into the actual queue those alerts would be
routed to for analyst review, mirroring DeepCASE's own semi-automatic mode
rather than just a conceptual description.


In [ ]:
manual_review_df.to_csv("grouping_manual_review_queue.csv", index=False)
print(f"Manual review queue: {len(manual_review_df)} alerts")
if not manual_review_df.empty:
    print(manual_review_df["reason"].value_counts())


## 9. Combined results and final comparison plots

Everything loaded above, combined into one DataFrame.


In [ ]:
all_results_df.to_csv("grouping_comparison_results.csv", index=False)

if all_results_df.empty:
    print("[skip] no results loaded -- run the baseline scripts first")
    summary_table = None
else:
    # Structural finding, not a trivial case: DeepCASE is the only method in
    # this comparison with a rejection/deferral path, so it should be the
    # only method with a nonzero ungroupable_fraction -- worth asserting, not
    # just omitting as an obvious zero.
    non_deepcase_ungroupable = all_results_df.loc[
        all_results_df["method"] != "deepcase", "ungroupable_fraction"
    ]
    assert (non_deepcase_ungroupable == 0.0).all(), (
        "Expected ungroupable_fraction == 0.0 for every non-DeepCASE method "
        "(none of them have a rejection step)"
    )
    print(
        "Confirmed: ungroupable_fraction is 0.0 for every non-DeepCASE method "
        "-- DeepCASE is the only method here with a rejection/deferral mode."
    )

    summary_table = all_results_df.groupby("method")[[
        "reduction", "coverage", "ungroupable_fraction", "group_size_mean", "n_groups",
        "pure_benign_frac", "pure_attack_frac", "mixed_frac",
        "host_purity", "signature_diversity_mean", "temporal_span_mean",
        "minority_label_exposure", "train_time_seconds", "inference_time_seconds",
    ]].mean()

summary_table


In [ ]:
if all_results_df.empty:
    print("[skip] no results loaded -- run the baseline scripts first")
else:
    # Final plot 1: reduction vs mean group size, all methods overlaid.
    # This is the key tradeoff plot -- high reduction from one giant group is
    # not the same achievement as high reduction from many well-formed groups.
    fig, ax = plt.subplots(figsize=(7, 5))

    for method_name, group in all_results_df.groupby("method"):
        agg = group.groupby("param", as_index=False)[["reduction", "group_size_mean"]].mean()
        ax.scatter(agg["group_size_mean"], agg["reduction"], label=method_name, alpha=0.7)

    ax.set_xscale("log")
    ax.set_xlabel("Mean group size (log scale)")
    ax.set_ylabel("Alert volume reduction")
    ax.set_title("Reduction vs. group size across all methods and settings")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if all_results_df.empty:
    print("[skip] no results loaded -- run the baseline scripts first")
else:
    # Final plot 2: purity comparison at each method's best-reduction setting
    # (or its only setting, for CSCAS). Pick the operating point per method,
    # then compare purity fractions side by side.
    best_settings = (
        all_results_df.sort_values("reduction", ascending=False)
        .groupby("method", as_index=False)
        .first()
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(best_settings))
    width = 0.25

    for i, col in enumerate(["pure_benign_frac", "pure_attack_frac", "mixed_frac"]):
        ax.bar(x + i * width, best_settings[col], width=width, label=col)

    ax.set_xticks(x + width)
    ax.set_xticklabels(best_settings["method"], rotation=20, ha="right")
    ax.set_ylabel("Fraction of groups")
    ax.set_title("Group purity at each method's highest-reduction setting")
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if all_results_df.empty:
    print("[skip] no results loaded -- run the baseline scripts first")
else:
    # Final plot 3: group size distributions side by side (log-y boxplot) at
    # each method's best-reduction setting -- surfaces the runaway-group
    # pathology (long streams of dense, repetitive alerts collapsing into a
    # single giant group) that flat mean/median stats can hide. Needs the
    # raw per-setting arrays from *_sizes.npz, not just the mean/median/max
    # already in all_results_df.
    fig, ax = plt.subplots(figsize=(9, 5))

    box_data = []
    labels = []
    for _, row in best_settings.iterrows():
        key = (row["method"], row["param"], row["scenario"])
        if key in size_arrays:
            box_data.append(size_arrays[key])
            labels.append(row["method"])

    if box_data:
        ax.boxplot(box_data, labels=labels, showfliers=True)
        ax.set_yscale("log")
        ax.set_ylabel("Group size (log scale)")
        ax.set_title("Group size distribution at each method's best-reduction setting")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("[skip] no group-size arrays loaded for any best-reduction setting")
